In [1]:
import pandas as pd
import numpy as np
import os

In [ ]:
#Import the top BPM and WPM tables
#These are for BPMs and WPMs that met a FDR threshold of 0.05 for at least one of the runs
top_bpms = pd.read_excel('C:/Users/sophi/Documents/Plan B Project/k10 new results/bpm_fdr_0.05.xlsx', usecols= 'B:D')
top_wpms = pd.read_excel('C:/Users/sophi/Documents/Plan B Project/k10 new results/wpm_fdr_0.05.xlsx', usecols = 'B:C')

In [ ]:
#Generate nums based on the number of case/control groups used for BridGE runs.
nums = np.arange(1,11)

In [56]:
#Combine the BPMs to make one giant excel spreadsheet
dfs = []
bpm_target = 'output_bpm_table'
for i in nums:
    xls = pd.ExcelFile(f'C:/Users/sophi/Documents/Plan B Project/CRC_g{i} results/output_results_with_runs.xlsx')
    #make sure sheet exists
    if bpm_target in xls.sheet_names:
        df = pd.read_excel(f'C:/Users/sophi/Documents/Plan B Project/CRC_g{i} results/output_results_with_runs.xlsx', sheet_name=bpm_target)
        dfs.append(df)

#Put all the dfs together
combined_df = pd.concat(dfs, ignore_index=True)

In [ ]:
#rename combined_df columns, so you can merge them later
combined_df = combined_df.rename(columns={'path1names':'path1', 'path2names':'path2', 'eff_bpm':'disease association'})

key_cols = ["disease association", "path1", "path2"]

# rows in combined that match those top BPMs
matched = combined_df.merge(top_bpms, on=key_cols, how="inner")

#Write to output
matched.to_csv('Output_Results_Table_BPM_FDR_0.05.csv')

In [ ]:
#Find the ones that were missed
merged = combined_df.merge(top_bpms, on=key_cols, how="outer", indicator=True)
not_found = merged[merged['_merge'] == 'right_only']

In [25]:
#match the name to the combined_df to extract the info here
missed_df = pd.DataFrame(columns= combined_df.columns)
for i in range(not_found.shape[0]):
    missed_val =  combined_df[(combined_df['path1'] == (not_found['path1'].iloc[i])) & (combined_df['path2'] == (not_found['path2'].iloc[i]))]
    missed_df = pd.concat([missed_df, missed_val], ignore_index=True)

C:\Users\sophi\AppData\Local\Temp\ipykernel_36300\159341012.py:5: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  missed_df = pd.concat([missed_df, missed_val], ignore_index=True)


In [26]:
missed_df

,Unnamed: 0,path1,path2,group,fdrBPM,disease association,size,bpm_pv,bpm_ranksum,bpm_path1_drivers,bpm_path2_drivers,Run_Num
0,3,REACTOME_TRANSPORT_OF_INORGANIC_CATIONS_ANIONS...,REACTOME_SIGNALING_BY_NUCLEAR_RECEPTORS,3,0.03,protective,33800,0.0035,39.36,rs28523687_SLC7A8_fold1.9_hyge2.89;rs10767571_...,rs8039428_IGF1R_fold2.22_hyge1.41;rs4921325_FA...,3
1,1,WP_BRAINDERIVED_NEUROTROPHIC_FACTOR_BDNF_SIGNA...,REACTOME_SIGNALING_BY_NUCLEAR_RECEPTORS,0,0.00,protective,46774,0.0043,43.93,rs3935700_IKBKB_fold1.79_hyge2.21;rs6077065_BM...,rs112892996_KANK1_fold3.64_hyge1.61;rs3760469_...,3
2,1,WP_COMPLEMENT_SYSTEM,REACTOME_EPIGENETIC_REGULATION_OF_GENE_EXPRESSION,1,0.00,protective,20370,0.0008,31.28,rs4958999_F12_fold2.2_hyge1.38;rs462477_C9_fol...,rs740851_CHD4_fold4.13_hyge1.79;rs3844213_HCFC...,7
3,0,WP_HAIR_FOLLICLE_DEVELOPMENT_CYTODIFFERENTIATI...,REACTOME_EPIGENETIC_REGULATION_OF_GENE_EXPRESSION,0,0.00,protective,19646,0.0005,37.30,rs10804243_IGFBP5_fold1.89_hyge2.79;rs8100483_...,rs7850942_EHMT1_fold2.67_hyge2.38;rs61421370_M...,7
4,5,WP_HAIR_FOLLICLE_DEVELOPMENT_CYTODIFFERENTIATI...,REACTOME_REGULATION_OF_INSULIN_LIKE_GROWTH_FAC...,2,0.03,protective,13500,0.0104,30.81,rs17108817_SPINK6_fold2.53_hyge4.81;rs1563557_...,rs6808800_KNG1_fold2.45_hyge1.79;rs540599_PAPP...,3
5,0,WP_OREXIN_RECEPTOR_PATHWAY,REACTOME_SIGNALING_BY_NUCLEAR_RECEPTORS,1,0.00,protective,59290,0.0019,60.23,rs2710995_OSBPL3_fold2.03_hyge3.18;rs7002316_T...,rs11047824_KRAS_fold2.62_hyge3.34;rs12205375_E...,3


In [ ]:
#now repeat for WPMs
#Combine the BPMs to make one giant excel spreadsheet
dfs = []
wpm_target = 'output_wpm_table'
for i in nums:
    xls = pd.ExcelFile(f'C:/Users/sophi/Documents/Plan B Project/CRC_g{i} results/output_results_with_runs.xlsx')
    #make sure sheet exists
    if wpm_target in xls.sheet_names:
        df = pd.read_excel(f'C:/Users/sophi/Documents/Plan B Project/CRC_g{i} results/output_results_with_runs.xlsx', sheet_name=wpm_target)
        dfs.append(df)

#Put all the dfs together
combined_df = pd.concat(dfs, ignore_index=True)

#rename combined_df columns, so you can merge them later
combined_df = combined_df.rename(columns={'eff_wpm':'disease association'})

key_cols = ["disease association", "pathway"]

# rows in combined that match those top WPMs
matched = combined_df.merge(top_wpms, on=key_cols, how="inner")

matched.to_csv("Output_Results_Table_WPM_FDR_0.05.csv")

In [33]:
#open the combined_pv_fdr files to extract the fdr values that we are looking for
comb_bpm = pd.read_excel('C:/Users/sophi/Documents/Plan B Project/k10 new results/combined_bpm_fdr_pv.xlsx')
comb_wpm = pd.read_excel('C:/Users/sophi/Documents/Plan B Project/k10 new results/combined_WPM_fdr_pv.xlsx')

In [63]:
#compare it to the matched dataframe from the earlier BPM step for this
bpm_fdrs = pd.DataFrame(columns=comb_bpm.columns)
for i in range(matched.shape[0]):
    the_fdrs =  comb_bpm[(comb_bpm['path1'] == (matched['path1'].iloc[i])) & (comb_bpm['path2'] == (matched['path2'].iloc[i]))]
    bpm_fdrs = pd.concat([bpm_fdrs, the_fdrs], ignore_index=True)

C:\Users\sophi\AppData\Local\Temp\ipykernel_36300\1799656154.py:5: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  bpm_fdrs = pd.concat([bpm_fdrs, the_fdrs], ignore_index=True)


In [ ]:
#find min_fdrs
min_fdrs_bpms = []
for i in range(matched.shape[0]):
    run_val = matched['Run_Num'].iloc[i]
    min_fdrs = bpm_fdrs[f'CRC_g{run_val}_fdr'].iloc[i]
    min_fdrs_bpms.append(min_fdrs)

In [68]:
#now do it for the ones you missed
#compare it to the matched dataframe from the earlier BPM step for this
bpm_fdrs = pd.DataFrame(columns=comb_bpm.columns)
for i in range(missed_df.shape[0]):
    the_fdrs =  comb_bpm[(comb_bpm['path1'] == (missed_df['path1'].iloc[i])) & (comb_bpm['path2'] == (missed_df['path2'].iloc[i]))]
    bpm_fdrs = pd.concat([bpm_fdrs, the_fdrs], ignore_index=True)

#find min_fdrs
min_fdrs_bpms = []
for i in range(missed_df.shape[0]):
    run_val = missed_df['Run_Num'].iloc[i]
    min_fdrs = bpm_fdrs[f'CRC_g{run_val}_fdr'].iloc[i]
    min_fdrs_bpms.append(min_fdrs)

C:\Users\sophi\AppData\Local\Temp\ipykernel_36300\43550345.py:6: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  bpm_fdrs = pd.concat([bpm_fdrs, the_fdrs], ignore_index=True)


In [71]:
#repeat for WPMs
#compare it to the matched dataframe from the earlier WPM step for this
wpm_fdrs = pd.DataFrame(columns=comb_wpm.columns)
for i in range(matched.shape[0]):
    the_fdrs =  comb_wpm[comb_wpm['pathway'] == (matched['pathway'].iloc[i])]
    wpm_fdrs = pd.concat([wpm_fdrs, the_fdrs], ignore_index=True)

#find min_fdrs
min_fdrs_wpms = []
for i in range(matched.shape[0]):
    run_val = matched['Run_Num'].iloc[i]
    min_fdrs = wpm_fdrs[f'CRC_g{run_val}_fdr'].iloc[i]
    min_fdrs_wpms.append(min_fdrs)

C:\Users\sophi\AppData\Local\Temp\ipykernel_36300\2666085323.py:6: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  wpm_fdrs = pd.concat([wpm_fdrs, the_fdrs], ignore_index=True)
